In [8]:
from os import listdir
from typing import TypedDict

import numpy as np
import torch
from PIL import Image
from sklearn.preprocessing import StandardScaler

from dinosaw.linear_probe import (
    RampTypes,
    Regressor,
    gen_sample_mask,
    get_model as get_reg_model,
    get_ramp,
)
from dinosaw.utils import get_features
from dinosaw.wrappers import ModelTypes
from dinosaw.wrappers.register_models import get_model_

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [9]:
selected_model: ModelTypes = "alibi_coco_dinov2_s"
model = get_model_(selected_model, DEVICE, True, "../../models/checkpoints", conf_dir="../../models/dinov3")
models = {selected_model: model}

n_dims = model.embed_dim

2026-11-09 21:59:47 | I | factory.py                 : 180 | Building wrapper 'alibi_coco_dinov2_s' on device cuda:0
2026-11-09 21:59:47 | I | factory.py                 : 158 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=False, checkpoint_path='../../models/checkpoints/trained/alibi_coco_dv2_vits14_reg4.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=True, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[functools.partial(<function add_alibi at 0x7d4f204a4cc0>, slope_type='constant', n_reg_tokens=4, metric='euclidean', normalize=True, wrap=True, add_cls=True, jitter_mag=0.0)], dtype=torch.float32)


VisionTransformer doesn't satisfy Dv3StyleViT; missing: n_storage_tokens, rope_embed


['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_init_weights', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_pos_embed', '_r

In [10]:
ds_folder = 'data/linear_probe/homog_micros'
MAX_IMGS: int | None = 60  # Set to 30 for the 30-image subset, or None to use all images in ds_folder
IMG_SIZE: tuple[int, int] = (518, 518)  # Resize so train/test patch masks align across images A and B
replace_with_random_noise: bool = False


def featurise_dataset(
    folder: str,
    max_imgs: int | None = MAX_IMGS,
    img_size: tuple[int, int] = IMG_SIZE,
    random_noise: bool = replace_with_random_noise,
) -> list[np.ndarray]:
    image_files = sorted(listdir(folder))
    if max_imgs is not None:
        image_files = image_files[:max_imgs]

    ds_features: list[np.ndarray] = []
    for img_file in image_files:
        img_path = f'{folder}/{img_file}'
        img = Image.open(img_path).convert('RGB').resize(img_size)
        if random_noise:
            noise_arr = np.random.randint(0, 256, (*img_size, 3), dtype=np.uint8)
            img = Image.fromarray(noise_arr).convert('RGB')
        feats = get_features(model, img, channel_last=True)
        ds_features.append(np.nan_to_num(feats, nan=0.0))
    return ds_features


features = featurise_dataset(ds_folder)
n_imgs = len(features)

2026-11-09 21:59:47 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:47 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:47 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:47 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:47 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:47 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:47 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:47 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:47 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:47 | I | wrapper.py                 : 2

In [11]:
class CrossImageProbeResult(TypedDict):
    # R^2 scores across the 4 evaluation sets:
    # 1) A_train: train on A, apply to same train locations in A
    # 2) A_test:  train on A, apply to test locations in A
    # 3) B_train: train on A, apply to same train locations in B
    # 4) B_test:  train on A, apply to test locations in B
    a_train_r2: list[float]
    a_test_r2: list[float]
    b_train_r2: list[float]
    b_test_r2: list[float]
    a_preds: list[np.ndarray]
    b_preds: list[np.ndarray]
    per_channel_a_train: list[np.ndarray] | None
    per_channel_a_test: list[np.ndarray] | None
    per_channel_b_train: list[np.ndarray] | None
    per_channel_b_test: list[np.ndarray] | None


def _eval_per_channel_r2(
    Z: np.ndarray,
    y_true: np.ndarray,
    w: np.ndarray,
    y_mean: np.ndarray,
) -> np.ndarray:
    """Vectorized evaluation of 1D per-channel linear probes.

    Z: (n_patches, c), y_true: (n_patches, d), w: (c, d), y_mean: (d,)
    Returns: per-channel R^2 array of shape (c,).
    """
    pred = Z[:, :, None] * w[None, :, :] + y_mean[None, None, :]  # (n_patches, c, d)
    ss_res = ((y_true[:, None, :] - pred) ** 2).sum(axis=0)  # (c, d)
    ss_tot = ((y_true - y_true.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)[None, :]  # (1, d)
    return (1.0 - ss_res / ss_tot).mean(axis=-1)


def do_cross_image_linear_probe(
    src_dataset: list[np.ndarray],
    ramp_type: RampTypes,
    targ_dataset: list[np.ndarray] | None = None,
    mask_step: int = 6,
    mask_cutoff_frac: float = 1.0,
    random_mask: bool = True,
    probe_by_channel: bool = True,
    scale_feats: bool = True,
    regressor: Regressor = "ridge",
    all_to_all: bool = False,
) -> CrossImageProbeResult:
    """Train a linear probe once per image A in `src_dataset` and evaluate on A (train/test) and B in `targ_dataset` (train/test).

    If `targ_dataset` is None, defaults to `src_dataset` (within-dataset cross-image probe).
    If `all_to_all=False`, pairs each image i with j = (i + 1) % N_targ (quick option).
    If `all_to_all=True`, evaluates each probe trained on image i across all j in `targ_dataset` (excluding j == i when within-dataset).
    """
    same_dataset = targ_dataset is None or targ_dataset is src_dataset
    if targ_dataset is None:
        targ_dataset = src_dataset

    n_src_imgs = len(src_dataset)
    n_targ_imgs = len(targ_dataset)
    h, w, c = src_dataset[0].shape
    target = get_ramp(ramp_type, h, w)
    d = target.shape[-1]

    a_train_r2: list[float] = []
    a_test_r2: list[float] = []
    b_train_r2: list[float] = []
    b_test_r2: list[float] = []
    a_preds: list[np.ndarray] = []
    b_preds: list[np.ndarray] = []

    per_channel_a_train: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_a_test: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_b_train: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_b_test: list[np.ndarray] | None = [] if probe_by_channel else None

    for i in range(n_src_imgs):
        feats_A = src_dataset[i]
        sample_mask = gen_sample_mask(
            (h, w), ramp_type, step=mask_step, cutoff_frac=mask_cutoff_frac, random_mask=random_mask
        )

        X_A_train = feats_A[sample_mask, :]
        X_A_test = feats_A[~sample_mask, :]
        X_A_full = feats_A.reshape(-1, c)
        y_train = target[sample_mask, :]
        y_test = target[~sample_mask, :]

        if scale_feats:
            a_scaler = StandardScaler().fit(X_A_full)
            X_A_train_sc = a_scaler.transform(X_A_train)
            X_A_test_sc = a_scaler.transform(X_A_test)
            X_A_full_sc = a_scaler.transform(X_A_full)
        else:
            X_A_train_sc = X_A_train
            X_A_test_sc = X_A_test
            X_A_full_sc = X_A_full

        # Train once on Image A
        reg = get_reg_model(regressor)
        reg.fit(X_A_train_sc, y_train)

        # 1) Apply to train locations in A & 2) Apply to test locations in A
        a_train_r2.append(float(reg.score(X_A_train_sc, y_train)))
        a_test_r2.append(float(reg.score(X_A_test_sc, y_test)))
        a_preds.append(reg.predict(X_A_full_sc).reshape(h, w, d))

        # Optional per-channel probe trained once on Image A
        if probe_by_channel:
            alpha = 1.0 if regressor == "ridge" else 0.0
            y_mean = y_train.mean(axis=0)
            denom = (X_A_train_sc ** 2).sum(axis=0) + alpha
            w_ch = np.einsum("nc,nd->cd", X_A_train_sc, y_train - y_mean[None, :]) / denom[:, None]
            assert per_channel_a_train is not None and per_channel_a_test is not None
            per_channel_a_train.append(_eval_per_channel_r2(X_A_train_sc, y_train, w_ch, y_mean))
            per_channel_a_test.append(_eval_per_channel_r2(X_A_test_sc, y_test, w_ch, y_mean))

        # Target image(s) B
        if all_to_all:
            b_indices = [j for j in range(n_targ_imgs) if not (same_dataset and j == i)]
        else:
            b_indices = [(i + 1) % n_targ_imgs]
        b_pred_accum = np.zeros((h, w, d), dtype=np.float64)

        for j in b_indices:
            feats_B = targ_dataset[j]
            X_B_train = feats_B[sample_mask, :]
            X_B_test = feats_B[~sample_mask, :]
            X_B_full = feats_B.reshape(-1, c)

            if scale_feats:
                b_scaler = StandardScaler().fit(X_B_full)
                X_B_train_sc = b_scaler.transform(X_B_train)
                X_B_test_sc = b_scaler.transform(X_B_test)
                X_B_full_sc = b_scaler.transform(X_B_full)
            else:
                X_B_train_sc = X_B_train
                X_B_test_sc = X_B_test
                X_B_full_sc = X_B_full

            # 3) Apply to same train locations in B & 4) Apply to test locations in B
            b_train_r2.append(float(reg.score(X_B_train_sc, y_train)))
            b_test_r2.append(float(reg.score(X_B_test_sc, y_test)))
            b_pred_accum += reg.predict(X_B_full_sc).reshape(h, w, d) / len(b_indices)

            if probe_by_channel:
                assert per_channel_b_train is not None and per_channel_b_test is not None
                per_channel_b_train.append(_eval_per_channel_r2(X_B_train_sc, y_train, w_ch, y_mean))
                per_channel_b_test.append(_eval_per_channel_r2(X_B_test_sc, y_test, w_ch, y_mean))

        b_preds.append(b_pred_accum)

    return {
        "a_train_r2": a_train_r2,
        "a_test_r2": a_test_r2,
        "b_train_r2": b_train_r2,
        "b_test_r2": b_test_r2,
        "a_preds": a_preds,
        "b_preds": b_preds,
        "per_channel_a_train": per_channel_a_train,
        "per_channel_a_test": per_channel_a_test,
        "per_channel_b_train": per_channel_b_train,
        "per_channel_b_test": per_channel_b_test,
    }

In [12]:
ONLY_LR_UD: bool = False  # Set True to only run 'lr+ud'; False to run ('lr', 'ud', 'diag', 'radial') + 'lr+ud'
ALL_TO_ALL: bool = False  # Set False for quick i -> (i+1)%N over the N images; True for all-to-all pairs (i, j)
SCALE_FEATS: bool = True
MASK_CUTOFF_FRAC = 1.0
STEP = 6
RANDOM_MASK = True

fig_ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramps_to_run: tuple[RampTypes, ...] = ('lr+ud',) if ONLY_LR_UD else (*fig_ramps, 'lr+ud')
ramps_to_results: dict[RampTypes, CrossImageProbeResult] = {}

mode_str = f"all-to-all ({n_imgs * (n_imgs - 1)} pairs)" if ALL_TO_ALL else f"i -> j ({n_imgs} pairs)"
print(f"Running cross-image linear probe [{mode_str}] on {n_imgs} images:")
print(f"{'Ramp':<8} | {'1) A_train':<15} | {'2) A_test':<15} | {'3) B_train':<15} | {'4) B_test':<15}")
print("-" * 78)

for ramp in ramps_to_run:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    res = do_cross_image_linear_probe(
        features,
        ramp,
        targ_dataset=features,
        mask_step=STEP,
        mask_cutoff_frac=MASK_CUTOFF_FRAC,
        random_mask=RANDOM_MASK,
        probe_by_channel=(ramp in fig_ramps and not ONLY_LR_UD),
        scale_feats=SCALE_FEATS,
        regressor="ridge",
        all_to_all=ALL_TO_ALL,
    )
    ramps_to_results[ramp] = res

    a_tr_m, a_tr_s = np.mean(res['a_train_r2']), np.std(res['a_train_r2'])
    a_te_m, a_te_s = np.mean(res['a_test_r2']), np.std(res['a_test_r2'])
    b_tr_m, b_tr_s = np.mean(res['b_train_r2']), np.std(res['b_train_r2'])
    b_te_m, b_te_s = np.mean(res['b_test_r2']), np.std(res['b_test_r2'])
    print(
        f"{ramp:<8} | "
        f"{a_tr_m:6.2f} ± {a_tr_s:5.2f} | "
        f"{a_te_m:6.2f} ± {a_te_s:5.2f} | "
        f"{b_tr_m:6.2f} ± {b_tr_s:5.2f} | "
        f"{b_te_m:6.2f} ± {b_te_s:5.2f}"
    )

Running cross-image linear probe [i -> j (60 pairs)] on 60 images:
Ramp     | 1) A_train      | 2) A_test       | 3) B_train      | 4) B_test      
------------------------------------------------------------------------------
lr       |   1.00 ±  0.01 |  -0.15 ±  0.51 |  -1.04 ±  1.25 |  -0.91 ±  0.85
ud       |   1.00 ±  0.00 |   0.16 ±  0.46 |  -0.54 ±  0.85 |  -0.52 ±  0.68
diag     |   1.00 ±  0.00 |   0.00 ±  0.54 |  -0.78 ±  1.03 |  -0.70 ±  0.84
radial   |   1.00 ±  0.00 |   0.49 ±  0.29 |  -0.30 ±  0.53 |  -0.29 ±  0.54
lr+ud    |   1.00 ±  0.00 |   0.00 ±  0.41 |  -0.79 ±  0.88 |  -0.71 ±  0.67


In [13]:
targ_ds_folder = 'data/linear_probe/texture_ds'
targ_features = featurise_dataset(targ_ds_folder)
n_targ_imgs = len(targ_features)

2026-11-09 21:59:53 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:53 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:53 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:53 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:53 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:53 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:53 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:53 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-09 21:59:53 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-09 21:59:53 | I | wrapper.py                 : 2

In [14]:
cross_ds_results: dict[RampTypes, CrossImageProbeResult] = {}

n_pairs = n_imgs * n_targ_imgs if ALL_TO_ALL else n_imgs
mode_str = f"all-to-all ({n_pairs} pairs)" if ALL_TO_ALL else f"i -> j ({n_pairs} pairs)"
print(f"Running cross-dataset linear probe [{ds_folder} -> {targ_ds_folder}, {mode_str}]:")
print(f"{'Ramp':<8} | {'1) A_train':<15} | {'2) A_test':<15} | {'3) B_train':<15} | {'4) B_test':<15}")
print("-" * 78)

for ramp in ramps_to_run:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    res = do_cross_image_linear_probe(
        features,
        ramp,
        targ_dataset=targ_features,
        mask_step=STEP,
        mask_cutoff_frac=MASK_CUTOFF_FRAC,
        random_mask=RANDOM_MASK,
        probe_by_channel=(ramp in fig_ramps and not ONLY_LR_UD),
        scale_feats=SCALE_FEATS,
        regressor="ridge",
        all_to_all=ALL_TO_ALL,
    )
    cross_ds_results[ramp] = res

    a_tr_m, a_tr_s = np.mean(res['a_train_r2']), np.std(res['a_train_r2'])
    a_te_m, a_te_s = np.mean(res['a_test_r2']), np.std(res['a_test_r2'])
    b_tr_m, b_tr_s = np.mean(res['b_train_r2']), np.std(res['b_train_r2'])
    b_te_m, b_te_s = np.mean(res['b_test_r2']), np.std(res['b_test_r2'])
    print(
        f"{ramp:<8} | "
        f"{a_tr_m:6.2f} ± {a_tr_s:5.2f} | "
        f"{a_te_m:6.2f} ± {a_te_s:5.2f} | "
        f"{b_tr_m:6.2f} ± {b_tr_s:5.2f} | "
        f"{b_te_m:6.2f} ± {b_te_s:5.2f}"
    )

Running cross-dataset linear probe [data/linear_probe/homog_micros -> data/linear_probe/texture_ds, i -> j (60 pairs)]:
Ramp     | 1) A_train      | 2) A_test       | 3) B_train      | 4) B_test      
------------------------------------------------------------------------------
lr       |   1.00 ±  0.01 |  -0.15 ±  0.51 |  -0.69 ±  0.89 |  -0.74 ±  0.77
ud       |   1.00 ±  0.00 |   0.16 ±  0.46 |  -0.42 ±  0.73 |  -0.38 ±  0.62
diag     |   1.00 ±  0.00 |   0.00 ±  0.54 |  -0.61 ±  0.87 |  -0.57 ±  0.73
radial   |   1.00 ±  0.00 |   0.49 ±  0.29 |  -0.19 ±  0.49 |  -0.21 ±  0.53
lr+ud    |   1.00 ±  0.00 |   0.00 ±  0.41 |  -0.55 ±  0.65 |  -0.56 ±  0.56
